# IAI600 – Lab V: Unsupervised Learning and K-Means Clustering

**Complete solution based on the supplied Lab V handout**

This notebook follows Parts I–IX in order. It includes runnable Python code, explanations, and answers to every interpretation question (Tasks 1.1–9.2).

## Learning objectives
- Explore Iris data without using target labels during clustering.
- Generate synthetic clusters and train K-Means.
- Understand centroids, assignments, and decision boundaries.
- Evaluate clustering using inertia.
- Explore initialization, Mini-Batch K-Means, and the elbow method.
- Apply K-Means to Iris and compare clusters with known species after training.

> **Important:** Cluster IDs (0, 1, 2) are arbitrary identifiers. They do not automatically correspond to Iris species labels.


## Setup

If needed, install dependencies with `%pip install numpy matplotlib scikit-learn`.

In [ ]:
# Uncomment if packages are missing:
# %pip install numpy matplotlib scikit-learn

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, make_blobs
from sklearn.cluster import KMeans, MiniBatchKMeans




## Part I – Loading and Exploring the Iris Dataset

The Iris dataset contains 150 observations and four numerical features. `X` contains measurements; `y` contains known species labels. For clustering, fit only on `X`.

In [ ]:
iris = load_iris()
X = iris.data
y = iris.target

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
print("Feature names:", iris.feature_names)
print("Species names:", iris.target_names)


### Explanation
- `X`: four measurements for each flower.
- `y`: known species label; do not use it to train clustering.
- `X.shape` is `(150, 4)`: 150 flowers and four features.
- `y.shape` is `(150,)`: one label per flower.

### Task 1.1 – Answers

1. **How many observations?** 150 flower samples.
2. **How many features per observation?** Four: sepal length, sepal width, petal length, and petal width.
3. **Why not use the target in unsupervised clustering?** Unsupervised learning discovers patterns without predefined target labels. Using `y` to guide training would change the intended experiment.
4. **What if known species labels were used to train a model?** This would move toward supervised learning; a classification model could learn from known species labels to predict predefined classes.


## Part II – Creating Synthetic Data

`make_blobs` generates artificial groups. It also returns `y_blobs`, but K-Means will be trained only with `X_blobs`.

In [ ]:
X_blobs, y_blobs = make_blobs(
    n_samples=300,
    centers=3,
    cluster_std=1.0,
    random_state=42
)

plt.figure(figsize=(7, 5))
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], s=20)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Synthetic Data")
plt.show()


### Parameter explanation
- `n_samples=300`: generates 300 observations.
- `centers=3`: generates three underlying groups.
- `cluster_std=1.0`: controls spread around generated centers; larger values generally create wider, more overlapping groups.
- `random_state=42`: makes generated data reproducible.

### Task 2.1 – Answers

1. **How many groups appear?** Three.
2. **What makes them distinguishable?** Points form separate concentrations in different locations.
3. **Why easier to visualize than Iris?** It has two features, so both can be plotted in 2D. Iris has four features.
4. **What does `cluster_std` control?** The spread of observations around centers; larger values generally create wider and potentially more overlapping groups.


## Part III – K-Means Clustering

K-Means repeatedly assigns each observation to its nearest center and updates each center using the mean of assigned observations. It repeats until convergence or a stopping condition.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(X_blobs)

print("Cluster centers:")
print(kmeans.cluster_centers_)

cluster_labels = kmeans.predict(X_blobs)
print("First 20 assignments:", cluster_labels[:20])
print("Training assignments from labels_:", kmeans.labels_[:20])


### Explanation
- `n_clusters=3`: request three clusters.
- `random_state=42`: reproducible initialization.
- `n_init=10`: tries multiple initializations and retains the best according to inertia.
- `fit(X_blobs)`: trains on observations.
- `cluster_centers_`: learned centroid coordinates.
- `predict()`: assigns observations to nearest centroids.
- `labels_`: assignments for training observations.

### Task 3.1 – Answers

1. **What do 0, 1, 2 represent?** Three cluster identifiers.
2. **Do they have inherent meaning?** No; they are arbitrary identifiers.
3. **Why is cluster 0 not automatically the first class?** K-Means does not know original class names; 0 identifies one discovered group.
4. **What does `cluster_centers_` represent?** Coordinates of centroids, each calculated as the mean position of observations assigned to that cluster.


## Part IV – Visualizing Clusters and Decision Boundaries

### Task 4.1 – Plot observations and learned centers

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=cluster_labels, s=25)
plt.scatter(
    kmeans.cluster_centers_[:, 0],
    kmeans.cluster_centers_[:, 1],
    marker="X", s=200, edgecolor="black"
)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("K-Means Clustering")
plt.show()


### Task 4.1 – Answers

1. **Where are centers located?** Near the middle of their groups, at the mean feature coordinates.
2. **Which observations belong to each cluster?** Each is assigned to the cluster with the nearest centroid in Euclidean distance.
3. **Are all observations equally close to their center?** No. Some are closer and some farther away; K-Means minimizes total squared distance, not each individual distance.
4. **Are some observations close to boundaries?** Yes. Points between neighboring groups may be close to a boundary and small changes can change their assignments.

### Task 4.2 – Plot decision boundaries

The grid covers the feature space. `predict()` assigns each grid position to its nearest centroid, and `contourf()` displays the resulting regions.

In [ ]:
x_min, x_max = X_blobs[:, 0].min() - 1, X_blobs[:, 0].max() + 1
y_min, y_max = X_blobs[:, 1].min() - 1, X_blobs[:, 1].max() + 1

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 500),
    np.linspace(y_min, y_max, 500)
)
grid = np.c_[xx.ravel(), yy.ravel()]
Z = kmeans.predict(grid).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.2)
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=cluster_labels, s=20)
plt.scatter(
    kmeans.cluster_centers_[:, 0],
    kmeans.cluster_centers_[:, 1],
    marker="X", s=200, edgecolor="black"
)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("K-Means Decision Boundaries")
plt.show()


### Task 4.2 – Answers

1. **What does each region represent?** Locations in feature space whose points would be assigned to the same centroid.
2. **How are boundaries related to centers?** They separate regions belonging to different centroids; between two centroids, the boundary consists of locations equally distant from both.
3. **Why are boundaries determined by distance?** K-Means assigns each observation to its nearest centroid.
4. **What if centroids move?** Boundaries move too, because distances to centroids change.


## Part V – Evaluating K-Means

Inertia is the sum of squared distances between observations and their assigned cluster centers:

\[
\text{Inertia} = \sum_{i=1}^{m} \left\|x^{(i)} - \mu_{c(i)}\right\|^2
\]

A lower inertia generally means observations are more compact around their assigned centroids.

In [ ]:
print("Inertia:", kmeans.inertia_)


### Task 5.1 – Answers

1. **What does inertia measure?** Total squared distance between observations and assigned centroids.
2. **Does smaller inertia generally indicate more compact clusters?** Yes; observations are, in total, closer to their assigned centroids.
3. **Why not use inertia alone to select cluster count?** Inertia generally decreases as clusters are added and can reach zero with one cluster per observation. Balance compactness with useful, simple grouping.


## Part VI – Random Initialization and Stability

Different initial centroid positions can lead to different final solutions. First, use one initialization (`n_init=1`) for each seed.

In [ ]:
for seed in [0, 1, 2, 3, 4]:
    model = KMeans(n_clusters=3, random_state=seed, n_init=1)
    model.fit(X_blobs)
    print("Random state:", seed, "Inertia:", model.inertia_)


### Task 6.1 – Answers

1. **Are results identical?** Not necessarily. Different seeds can produce different inertia and assignments, although this well-separated dataset may converge to the same solution.
2. **Why can initializations differ?** K-Means can converge to different local solutions depending on starting centroid positions.
3. **Which solution is preferred if inertia differs?** With the same data, features, and cluster count, the lower-inertia solution is generally preferred for compactness.
4. **Why is reproducibility important?** It allows experiments to be repeated, compared fairly, debugged, and interpreted.

### Multiple initializations with `n_init`

Compare one initialization with ten.

In [ ]:
kmeans_single = KMeans(n_clusters=3, random_state=42, n_init=1)
kmeans_single.fit(X_blobs)

kmeans_multi = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_multi.fit(X_blobs)

print("Inertia with n_init=1:", kmeans_single.inertia_)
print("Inertia with n_init=10:", kmeans_multi.inertia_)


### Task 6.2 – Answers

1. **Purpose of `n_init`?** Number of runs with different centroid initializations; the best inertia result is retained.
2. **Why can multiple initializations improve results?** They provide multiple chances to find a better solution rather than relying on one potentially poor start.
3. **Trade-off of increasing initializations?** More computation and training time, but potentially a lower-inertia solution.


## Part VII – Mini-Batch K-Means

Mini-Batch K-Means uses smaller subsets of observations during training. It can reduce computation for large datasets.

In [ ]:
mini_batch_kmeans = MiniBatchKMeans(
    n_clusters=3,
    random_state=42,
    batch_size=50,
    n_init=10
)
mini_batch_kmeans.fit(X_blobs)

print("Mini-Batch inertia:", mini_batch_kmeans.inertia_)
print("Standard K-Means inertia:", kmeans_multi.inertia_)
print("Mini-Batch K-Means inertia:", mini_batch_kmeans.inertia_)


### Task 7.1 – Answers

1. **Are inertia values exactly the same?** Not necessarily; Mini-Batch may have slightly higher or otherwise different inertia.
2. **Why might results differ?** It updates centroids using subsets, so its optimization process differs from standard K-Means.
3. **Why useful for very large datasets?** Smaller batches can reduce computation per update.
4. **Speed versus quality trade-off?** It can train faster, but may produce somewhat higher inertia or a less optimal solution.


## Part VIII – Finding the Optimal Number of Clusters

The elbow method calculates inertia for several values of `k` and looks for a point where adding clusters produces substantially smaller improvements.

In [ ]:
inertias = []

for k in range(1, 10):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_blobs)
    inertias.append(model.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 10), inertias, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Inertia vs. Number of Clusters")
plt.xticks(range(1, 10))
plt.grid(True)
plt.show()

for k, inertia in zip(range(1, 10), inertias):
    print(f"k={k}: inertia={inertia:.2f}")


### Task 8.1 – Answers

1. **What happens to inertia as `k` increases?** It generally decreases or stays the same.
2. **Why does it decrease?** More clusters allow smaller groups of observations to have their own centroids, reducing distances.
3. **Where is the elbow?** At `k=3` for this experiment: inertia falls sharply up to three clusters and improvements become much smaller afterward. This matches the three generated groups.
4. **Why not choose the largest `k`?** It may split natural groups into unnecessary smaller groups, making the result complicated and less useful.
5. **Why is the elbow method a heuristic?** Some curves have no clear elbow; choosing it can be subjective and depends on the data and application.


## Part IX – Applying K-Means to Iris

Train on all four Iris features. Do not use the known labels during training.

In [ ]:
iris_kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
iris_kmeans.fit(X)

iris_clusters = iris_kmeans.labels_

print("Cluster centers:")
print(iris_kmeans.cluster_centers_)
print("\nInertia:")
print(iris_kmeans.inertia_)


### Visualize two features

The lab selects petal length (`X[:, 2]`) and petal width (`X[:, 3]`) for a 2D plot. The model itself was trained on all four features.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 2], X[:, 3], c=iris_clusters, s=30)
plt.xlabel("Petal length")
plt.ylabel("Petal width")
plt.title("K-Means Clustering of Iris Data")
plt.show()


### Task 9.1 – Answers

1. **Do observations form distinct groups?** Yes. Petal measurements reveal visible groups. One group is particularly distinct, while the other groups overlap to some extent.
2. **Which features provide useful separation?** Petal length and petal width, because measurements differ substantially among some Iris species.
3. **Are clusters perfectly separated?** No. Some observations from different groups have similar petal measurements and overlap.
4. **Why are some observations difficult to assign?** Some flowers have similar measurements and lie close to cluster boundaries. K-Means still assigns each flower to one nearest centroid.

### Compare with known labels – only after clustering

Inspect the original labels after the unsupervised model has been trained.

In [ ]:
print("Original species:")
print(iris.target_names)

print("\nFirst 20 known labels:")
print(y[:20])

print("\nFirst 20 cluster assignments:")
print(iris_clusters[:20])


### Optional: contingency table

Rows are K-Means cluster IDs; columns are original species IDs. This is a post-training comparison, not an input to clustering.

In [ ]:
from sklearn.metrics import confusion_matrix

print("Rows: K-Means clusters")
print("Columns: original species labels")
print(confusion_matrix(iris_clusters, y))


### Task 9.2 – Answers

1. **How well do clusters correspond visually to known species?** They correspond reasonably well, especially for setosa, which is more distinct in petal measurements. Versicolor and virginica overlap more, so observations may be distributed across clusters.
2. **Why not compare cluster 0 directly with species label 0?** Cluster IDs are assigned by K-Means and have no inherent relationship to species labels. Cluster 0 does not automatically mean setosa; examine the observations in each cluster.
3. **Difference between clustering and classification?** Clustering is unsupervised and discovers groups from feature similarities without known target labels. Classification is supervised and learns from labeled examples to predict predefined classes. Clustering may discover groups resembling species but is not guaranteed to reproduce them perfectly.


## Final Summary

| Concept | Key point |
|---|---|
| Unsupervised learning | Finds patterns without target labels |
| `X` and `y` | Train clustering with `X`; keep `y` aside |
| K-Means | Assigns points to nearest centroids and updates centers |
| Centroid | Mean position of points assigned to a cluster |
| Inertia | Sum of squared distances to assigned centroids |
| `n_init` | Number of initialization attempts |
| Mini-Batch K-Means | Uses mini-batches to reduce computation |
| Elbow method | Heuristic for selecting a useful cluster count |
| Cluster IDs | Arbitrary identifiers, not original class labels |

**End of Lab V.** Run notebook cells from top to bottom. Numerical results can vary slightly with scikit-learn versions, particularly for Mini-Batch K-Means.
